# 16 — The capstone synthesis

In `15` you scored a quantum measure and its classical neighbour on ensembles drawn by hand, and you left with two limits: one pair of ensembles, and a second moment you turned yourself. Today you lift both. You meet a pair of coupled oscillators whose second harmonic moves on its own, you read what a **bench** measured on it — one model and one protocol, run once, with its results committed — and you write the accounting of the capstone from those measurements.

**Prerequisites:** `11_kuramoto_dyad_and_ground_truth` (the coupled pair), `13_the_redundancy_reveal`, `14_richer_embedding_applied` (the multi-frequency qutrit) and `15_discriminating_experiment` (the second-harmonic phase-locking value, the area under the curve and the detection rate).

**What you'll be able to do**
- Simulate a pair of oscillators coupled through **two harmonics** and check its sample moments against exact values.
- Read the committed results of the bench and reproduce one of its comparisons yourself.
- Say **how long a record** each statistic needs when the coupling lives in the second harmonic, and when the phase-locking value (PLV) is held equal.
- Show that the quantum measures computed here read the **relative phase** of the two harmonics only through its cosine.
- State what a quantum measure reads, what it does not, and the limits of the evidence.

In [ ]:
import csv
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum_ot.bench_protocol import compare_conditions, long_run_statistics
from qot_course.quantum_ot.biharmonic import (
    discretised_stationary_moments,
    long_run_joint_density,
    simulate_biharmonic_dyad,
)

np.random.seed(42)
viz.use_course_style()

# The bench was run once and its results are committed with the course. They sit at the
# repository root, which we find by walking up from this notebook to the project file.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
RESULTS = REPO_ROOT / "results" / "second_harmonic_bench"

metadata = json.loads((RESULTS / "run_metadata.json").read_text())
BENCH = metadata["config"]      # every setting of the run, as it was resolved
MODEL = BENCH["model"]          # omega_1, omega_2 (rad/s), noise_std, dt (s), burn_in_s (s)
PHYSICS = {key: MODEL[key] for key in ("omega_1", "omega_2", "noise_std")}

print("model of the bench:", MODEL)
print("records per set:", BENCH["protocol"]["n_records"],
      "| repetitions:", len(BENCH["protocol"]["master_seeds"]),
      "| record durations (s):", BENCH["protocol"]["durations_s"])

**Read the output.** The bench is one run of `scripts/run_second_harmonic_bench.py`, about eighteen minutes of computation, whose results are committed in `results/second_harmonic_bench/`. This notebook reads those files and simulates fresh records beside them, so that you can test what the files say against runs of your own. The lines above give the model it used and its protocol: 100 records per set, three record durations, five repetitions with different seeds.

## 1. A bench where the second harmonic is free

Why not run the experiment of `15` on the dyad of `11`? Because that dyad has one knob. Once its coupling $K$ is chosen, every moment of the phase difference is fixed with it, so a measure that reads the second harmonic has nothing of its own to show there.

The bench adds the next harmonic of the coupling, with its own strength $K_2$:

$$ \mathrm{d}\theta_1 = \bigl[\omega_1 + K_1 \sin(\theta_2 - \theta_1) + K_2 \sin 2(\theta_2 - \theta_1)\bigr]\,\mathrm{d}t + \sigma\,\mathrm{d}W_1 , $$

and the same for $\theta_2$ with the indices exchanged. With $K_2 = 0$ it is the dyad of `11`. Haken, Kelso and Bunz (1985) wrote this two-harmonic coupling for the relative phase of two fingers moving together; it is written here for two phase oscillators, and Schöner, Haken and Kelso (1986) added the noise.

We write $m_1 = \langle e^{i\Delta}\rangle$ and $m_2 = \langle e^{2i\Delta}\rangle$ for the first two circular moments of the phase difference $\Delta = \theta_1 - \theta_2$: these are the $a_1$ and $a_2$ of `15`, now complex. $|m_1|$ is the PLV and $|m_2|$ the second-harmonic PLV, written PLV2 in the tables. A **record** is one simulated run of the pair, after a warm-up of 50 s that is discarded. The bench also computes both moments **exactly**, with no sampling: it solves for the law that one time step of the simulated scheme leaves unchanged (`discretised_stationary_moments`). Let us simulate three of its settings, and the uncoupled pair, and compare.

In [ ]:
def solved(experiment, label):
    """The two coupling settings (K1, K2) of one comparison of the bench, as it ran them.

    The metadata file stores them as `condition_a` and `condition_b`, except for the
    comparison at equal moduli, which stores the pair it solved for under `pair`.
    """
    entry = next(s for s in metadata["solved_settings"]
                 if s["experiment"] == experiment and s["label"] == label)
    if "pair" in entry:
        pair = entry["pair"]
        return (pair["k1_ref"], pair["k2_ref"]), (pair["k1"], pair["k2"])
    return tuple(entry["condition_a"]), tuple(entry["condition_b"])


def simulate(k1, k2):
    """Phase difference of fresh records at couplings (K1, K2), shape (records, samples)."""
    theta_1, theta_2 = simulate_biharmonic_dyad(
        k1, k2, n_records=N_LIVE_RECORDS, duration=LIVE_DURATION, dt=MODEL["dt"],
        burn_in=MODEL["burn_in_s"], seed=16, **PHYSICS,
    )
    return theta_1 - theta_2


def exact_moments(k1, k2, max_harmonic=2):
    """Exact long-run moments (1, m1, m2, ...) of the simulated scheme, without sampling."""
    return discretised_stationary_moments(
        k1, k2, dt=MODEL["dt"], max_harmonic=max_harmonic, **PHYSICS
    )


def mean_and_error(values):
    """Mean over the records and its standard error, as text."""
    return f"{values.mean():>6.3f} ± {values.std(ddof=1) / np.sqrt(values.size):.3f}"


first_only, both = solved("equal_plv", "target=0.3,k2=-0.1")
_, second_only = solved("detection", "k2=0.1")
SETTINGS = {
    "first harmonic only": first_only,
    "second harmonic only": second_only,
    "both harmonics": both,
}

N_LIVE_RECORDS = 100     # records simulated here, each as long as the longest of the bench
LIVE_DURATION = 3200.0   # seconds

deltas = {}
print(f"{N_LIVE_RECORDS} records of {LIVE_DURATION:,.0f} s per setting; "
      "mean over records ± its standard error")
print(f"{'setting':<22}{'K1':>8}{'K2':>8}   "
      f"{'|m1| sample':>14}{'exact':>8}   {'|m2| sample':>14}{'exact':>8}")
for name, (k1, k2) in {"no coupling": (0.0, 0.0), **SETTINGS}.items():
    delta = simulate(k1, k2)
    deltas[name] = delta
    m1 = np.abs(np.exp(1j * delta).mean(axis=1))    # one modulus per record
    m2 = np.abs(np.exp(2j * delta).mean(axis=1))
    exact = exact_moments(k1, k2)
    print(f"{name:<22}{k1:>8.4f}{k2:>8.4f}   "
          f"{mean_and_error(m1):>14}{abs(exact[1]):>8.3f}   "
          f"{mean_and_error(m2):>14}{abs(exact[2]):>8.3f}")

**Read the output.** Wherever a harmonic is coupled, the sample modulus lands within two standard errors of the exact value: 0.298 for 0.300, 0.360 for 0.359, 0.368 for 0.365. The simulation and the exact computation agree, and you checked it yourself.

Two details matter later. The last setting has the **same PLV** as the first-harmonic one, 0.300, with a second moment five times larger: section 3 compares them. And where the exact moment is zero, the sample modulus is not: 0.028 without coupling, 0.037 with a second harmonic alone. A modulus is never negative, so the noise of a finite record does not average out of it. Keep this in mind for section 2.

In [ ]:
def load_rows(name):
    """Rows of one results file of the bench, as dictionaries of strings."""
    with open(RESULTS / name, newline="") as handle:
        return list(csv.DictReader(handle))


def law_from_moments(moments, angle):
    """Density of the phase difference rebuilt from its moments (a Fourier series), in 1/rad."""
    orders = np.arange(1, moments.size)[:, None]
    waves = np.real(moments[1:, None] * np.exp(-1j * orders * angle))
    return (1.0 + 2.0 * waves.sum(axis=0)) / (2.0 * np.pi)


grid = load_rows("map.csv")   # exact long-run moments on a grid of couplings (K1, K2)
grid_k2 = np.array([float(row["k2"]) for row in grid])
grid_m1 = np.array([float(row["abs_m1"]) for row in grid])
grid_m2 = np.array([float(row["abs_m2"]) for row in grid])
grid_qmi = np.array([float(row["qmi_rich_exact"]) for row in grid])

on_line = grid_k2 == 0.0   # the settings with no second harmonic in the coupling
order = np.argsort(grid_m1[on_line])
same_order = (np.array_equal(order, np.argsort(grid_m2[on_line]))
              and np.array_equal(order, np.argsort(grid_qmi[on_line])))
band = (grid_m1 >= 0.30) & (grid_m1 < 0.35)
line_low, line_high = np.interp([0.30, 0.35], grid_m1[on_line][order], grid_m2[on_line][order])
print(f"{len(grid):,} settings on the grid, {on_line.sum()} of them with K2 = 0")
print(f"with K2 = 0, |m1|, |m2| and the exact rich QMI put the {on_line.sum()} settings "
      f"in the same order: {same_order}")
weak = band & (np.abs(grid_k2) <= 0.15)   # the range of K2 the bench simulates records at
print(f"settings with |m1| between 0.30 and 0.35: {band.sum()}, whose |m2| runs from "
      f"{grid_m2[band].min():.3f} to {grid_m2[band].max():.3f}")
print(f"  of which {weak.sum()} have |K2| <= 0.15, whose |m2| runs from "
      f"{grid_m2[weak].min():.3f} to {grid_m2[weak].max():.3f}")
print(f"on the K2 = 0 line, |m2| goes from {line_low:.3f} to {line_high:.3f} "
      "over the same interval")

fig, (ax_law, ax_map) = plt.subplots(1, 2, figsize=(12, 4.8))

# (a) The law of the phase difference in the three settings: histogram of the live records
#     and, as a line, the exact law of the simulated scheme rebuilt from 30 moments.
colors = {"first harmonic only": COLORS["source"], "second harmonic only": COLORS["target"],
          "both harmonics": COLORS["quantum"]}
angle = np.linspace(-np.pi, np.pi, 361)
for name, (k1, k2) in SETTINGS.items():
    wrapped = np.angle(np.exp(1j * deltas[name])).ravel()
    ax_law.hist(wrapped, bins=72, range=(-np.pi, np.pi), density=True, histtype="stepfilled",
                color=colors[name], alpha=0.30, label=f"{name} (records)")
    ax_law.plot(angle, law_from_moments(exact_moments(k1, k2, max_harmonic=30), angle),
                color=colors[name], lw=1.8)
ax_law.plot([], [], color=COLORS["text"], lw=1.8, label="exact law")
ax_law.set_xlabel(r"phase difference $\Delta$ (rad)")
ax_law.set_ylabel("probability density (1/rad)")
ax_law.set_xticks([-np.pi, -np.pi / 2, 0.0, np.pi / 2, np.pi])
ax_law.set_xticklabels([r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"])
ax_law.set_ylim(0.0, 0.56)
ax_law.set_title("Three settings, three laws", pad=10)
ax_law.legend(loc="upper right", fontsize=8)

# (b) Every grid point of the bench in the plane of the two moduli.
ax_map.scatter(grid_m1[~on_line], grid_m2[~on_line], s=10, color=COLORS["muted"], alpha=0.45,
               edgecolor="none", label=r"$K_2 \neq 0$")
ax_map.plot(grid_m1[on_line][order], grid_m2[on_line][order], color=COLORS["highlight"], lw=2.6,
            label=r"$K_2 = 0$: the dyad of notebook 11")
for name, (k1, k2) in SETTINGS.items():
    exact = exact_moments(k1, k2)
    ax_map.scatter(abs(exact[1]), abs(exact[2]), s=90, color=colors[name], edgecolor="white",
                   linewidth=1.2, zorder=5, label=name)
ax_map.set_xlabel(r"$|m_1|$, the phase-locking value")
ax_map.set_ylabel(r"$|m_2|$, the second-harmonic phase-locking value")
ax_map.set_xlim(-0.02, 1.0)
ax_map.set_ylim(-0.02, 1.0)
ax_map.set_title("The second moment is free", pad=10)
ax_map.legend(loc="lower right", fontsize=8)

fig.suptitle("A pair of oscillators with two harmonics in its coupling", y=1.0)
fig.tight_layout()
plt.show()

**Read the figure.** On the left, each filled shape is the histogram of your records and each line the exact law: they coincide. A first harmonic alone gives one bump, away from zero because the two oscillators have different natural frequencies. A second harmonic alone gives **two equal bumps half a turn apart**: the law repeats every half turn, and its first moment is exactly zero. Both harmonics together give two unequal bumps.

On the right, each grey dot is one of the 1,066 coupling settings of the bench, placed by its exact PLV and second-harmonic PLV. The rose line holds the 26 settings with $K_2 = 0$. Along it the printout says that $|m_1|$, $|m_2|$ and the exact rich quantum mutual information (QMI) rank the settings identically: on the one-knob dyad, the richer embedding of `14` is still an echo of PLV. Off the line the second moment is free. For a PLV between 0.30 and 0.35 it runs from 0.19 to 0.47 at the coupling strengths the bench simulates ($|K_2| \le 0.15$), and up to 0.89 at stronger ones, where the line allows 0.07 to 0.10.

The three large dots are the settings you simulated, in the colours of the left panel (colours start afresh in each figure of this notebook). The amber one has a PLV of zero: opposed to the uncoupled pair, it is the **detection** comparison of section 2. The blue one stands straight above the violet one: same PLV, different second moment, the **equal-PLV** comparison of section 3.

## 2. Detection through the second harmonic

First question: a pair coupled through its second harmonic only, against a pair that is not coupled at all. The exact PLV is zero on both sides, so this is a coupling the phase-locking value cannot see in the long run.

Five statistics are scored on every record. PLV and PLV2 are the classical ones. The **naive QMI** is the quantum mutual information of the phase-qubit embedding of `12`, which carries the first harmonic only. The **rich QMI** and the **rich Bures** distance are the mutual information, in nats, and the Bures-coupling distance of `12`, both taken on the qutrit embedding of `14`, which carries two harmonics with equal weights.

The protocol is the one of `15`, with records in place of draws. The area under the curve (AUC) is the probability that a record of setting B scores higher than a record of setting A. The detection rate is the share of differences between B and A that fall outside a band holding 95 % of the differences between two sets of A. We read the rates from the bench files, where a label such as `k2=0.05` names the coupled setting, then rerun one comparison with a seed the bench never used.

In [ ]:
summary = load_rows("summary.csv")   # one row per comparison, record duration and statistic

NAMES = {
    "plv": "PLV",
    "plv2": "PLV2",
    "qmi_naive": "naive QMI",
    "qmi_rich": "rich QMI",
    "bures_rich": "rich Bures",
}
DURATIONS = BENCH["protocol"]["durations_s"]


def read(experiment, label, duration, statistic, column):
    """One number of summary.csv; NaN where the bench wrote an empty cell."""
    for row in summary:
        same_row = (row["experiment"], row["label"], row["statistic"]) == (
            experiment, label, statistic)
        if same_row and float(row["duration_s"]) == duration:
            return float(row[column]) if row[column] else float("nan")
    raise KeyError((experiment, label, duration, statistic))


def print_rates(experiment, labels, statistics, column="detection_rate_mean"):
    """Mean over the five repetitions of one column, one row per comparison and duration."""
    print(f"{'comparison':<22}{'record':>9}" + "".join(f"{NAMES[s]:>12}" for s in statistics))
    for label in labels:
        for duration in DURATIONS:
            cells = [read(experiment, label, duration, s, column) for s in statistics]
            print(f"{label:<22}{duration:>7.0f} s" + "".join(f"{c:>12.2f}" for c in cells))


print("Detection rate, uncoupled pair against a pair coupled by the second harmonic only")
print_rates("detection", ["k2=0.05", "k2=0.1", "k2=0.15"], list(NAMES))

# What the sample PLV does at 3,200 s, where its exact value is zero in both settings.
print()
print("Sample PLV on records of 3,200 s (its exact value is 0 on both sides)")
print(f"{'comparison':<12}{'mean, uncoupled':>17}{'mean, coupled':>15}"
      f"{'scatter, uncoupled':>20}{'scatter, coupled':>18}{'area':>7}")
for label in ("k2=0.05", "k2=0.1", "k2=0.15"):
    cells = [read("detection", label, 3200.0, "plv", column)
             for column in ("mean_a", "mean_b", "mean_null_sd_a", "mean_null_sd_b", "auc_mean")]
    print(f"{label:<12}{cells[0]:>17.3f}{cells[1]:>15.3f}{cells[2]:>20.3f}{cells[3]:>18.3f}"
          f"{cells[4]:>7.2f}")

# The same comparison, simulated here with a fresh seed: K2 = 0.1, records of 800 s.
uncoupled, coupled = solved("detection", "k2=0.1")
live_model = {**PHYSICS, "dt": MODEL["dt"], "burn_in": MODEL["burn_in_s"],
              "n_modes": BENCH["solver"]["n_modes"]}
live = compare_conditions(uncoupled, coupled, duration=800.0,
                          n_records=BENCH["protocol"]["n_records"], seed=16,
                          model=live_model, weight_sets={})
print()
print("K2 = 0.1, records of 800 s: detection rate simulated here, and in the committed files")
print(f"{'statistic':<12}{'here':>8}{'files: mean [min, max]':>28}")
for row in live:
    if row["statistic"] in NAMES:
        low, mean, high = (
            read("detection", "k2=0.1", 800.0, row["statistic"], f"detection_rate_{c}")
            for c in ("min", "mean", "max")
        )
        print(f"{NAMES[row['statistic']]:<12}{row['detection_rate']:>8.2f}"
              f"{f'{mean:.2f} [{low:.2f}, {high:.2f}]':>28}")

**Read the output.** The second-harmonic PLV detects the coupling first. At $K_2 = 0.05$ its detection rate is 0.29 on records of 200 s, 0.89 at 800 s and 1.00 at 3,200 s.

The rich quantum measures detect it too, and that deserves a pause: the exact PLV is zero here, and a quantum measure sees the coupling. They need longer records: in all three comparisons the rich QMI passes 0.8 one step of the grid after the second-harmonic PLV, and the rich Bures distance follows a little below. The grid moves by factors of four, so it brackets the delay loosely: at $K_2 = 0.05$ the ratio of the two durations lies between 1 and 16. In `15` the ratio was at least a hundred, on independent draws from a hand-written law. The ratio belongs to the pair of laws compared, as `15` warned.

PLV and the naive QMI stay low, yet not at 0.05: they climb to 0.36 and 0.38 at $K_2 = 0.15$. The second table shows why. The exact PLV is zero on both sides and the sample PLV is not: 0.031 without coupling and 0.056 with it, with a scatter that grows from 0.021 to 0.054. The modulus of a noisy mean is never negative, and the coupled pair, which lingers in each of its two relations, has the noisier mean. A reader of PLV alone would take it for a weak first harmonic, and there is none.

Your own run is the last table; the six log lines above it are the six sets of records a comparison draws, A and B and two more of each. It agrees with the files: 1.00, 0.98 and 0.88 for the three sighted statistics, inside the ranges of the five repetitions. Your PLV rate, 0.09, falls a hair under the smallest of the five: a sixth repetition can land a little outside their range.

## 3. Equal PLV

The question of the capstone is sharper: two coupled pairs with **the same PLV**, one without a second harmonic (A) and one with it (B). This is the experiment of `15`, carried to a system whose moments come out of its dynamics.

For each of four values of $K_2$, the bench solved for the first-harmonic coupling that gives B exactly the PLV of A, 0.3 or 0.6. A label such as `target=0.3,k2=-0.1` gives that shared PLV and the $K_2$ of B. A positive $K_2$ pulls the pair towards in phase or opposite, a negative one towards a quarter turn apart. The first row is the pair you simulated in section 1, and the last table gives the scatter of PLV within each setting.

In [ ]:
# Each comparison is labelled by the PLV both settings share and by the K2 of setting B.
EQUAL_PLV_LABELS = ["target=0.3,k2=-0.1", "target=0.3,k2=0.05", "target=0.6,k2=-0.15",
                    "target=0.6,k2=0.04"]

print("Exact long-run values of the two settings (A without, B with a second harmonic)")
print(f"{'comparison':<22}{'PLV A':>8}{'PLV B':>8}{'PLV2 A':>9}{'PLV2 B':>9}"
      f"{'rich QMI A':>12}{'rich QMI B':>12}")
for label in EQUAL_PLV_LABELS:
    cells = [read("equal_plv", label, 3200.0, s, f"exact_discrete_{side}")
             for s in ("plv", "plv2", "qmi_rich") for side in ("a", "b")]
    print(f"{label:<22}{cells[0]:>8.3f}{cells[1]:>8.3f}{cells[2]:>9.3f}{cells[3]:>9.3f}"
          f"{cells[4]:>12.4f}{cells[5]:>12.4f}")

print()
print("Area under the curve at 3,200 s")
print(f"{'comparison':<22}" + "".join(f"{NAMES[s]:>12}" for s in NAMES))
for label in EQUAL_PLV_LABELS:
    print(f"{label:<22}" + "".join(
        f"{read('equal_plv', label, 3200.0, s, 'auc_mean'):>12.2f}" for s in NAMES))

print()
print("Detection rate")
print_rates("equal_plv", EQUAL_PLV_LABELS, list(NAMES))

# Scatter of PLV within each setting: differences between two sets of A, and between two of B.
print()
print("Scatter of the PLV at 3,200 s (standard deviation of a difference between two records)")
for label in EQUAL_PLV_LABELS:
    within_a, within_b = (read("equal_plv", label, 3200.0, "plv", f"mean_null_sd_{side}")
                          for side in ("a", "b"))
    print(f"{label:<22}within A: {within_a:.3f}   within B: {within_b:.3f}")

**Read the output.** Start with the exact values. In every pair the PLV is equal to three decimals and the second-harmonic PLV is not. The exact rich QMI differs as well: 0.0408 against 0.0536 nat in the first pair. In the long run, the richer quantum measure **does** tell apart two coupled pairs that PLV cannot. The promise of `14` holds on a dynamical system.

On records of finite length the difference is harder to catch. The second-harmonic PLV passes a detection rate of 0.8 at 800 s in two pairs and at 3,200 s in a third. The rich QMI and the rich Bures distance pass it in none, at any duration: their best is 0.32, in the first pair at 3,200 s. Their AUC there is 0.83, so the shift is real and its direction is right; it is too small beside the scatter of one record to be detected reliably. The fourth pair, with the smallest $K_2$, defeats every statistic of the table: 0.67 is the best rate.

PLV and the naive QMI have an AUC between 0.46 and 0.52: chance. Their detection rates still reach 0.18, because the band is built on setting A alone and the rate counts any departure from it, including a wider scatter in B with no shift: 0.056 within B against 0.035 within A in the first pair, and wider in all four. The AUC, which asks only for a shift, is the safer reading here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)

styles = {
    "plv": (COLORS["muted"], "-"),
    "qmi_naive": (COLORS["target"], "--"),
    "plv2": (COLORS["source"], "-"),
    "qmi_rich": (COLORS["quantum"], "-"),
    "bures_rich": (COLORS["highlight"], ":"),
}
POWER_CONVENTION = BENCH["protocol"]["reliability_threshold"]   # 0.8, a convention
panels = [
    ("detection", "k2=0.05", r"Detection: $K_2 = 0.05$ against no coupling"),
    ("equal_plv", "target=0.3,k2=-0.1", r"Equal PLV of 0.3: $K_2 = -0.1$ against $K_2 = 0$"),
]
for ax, (experiment, label, title) in zip(axes, panels):
    ax.axhline(POWER_CONVENTION, color=COLORS["muted"], lw=1.2, ls=":", zorder=1,
               label="0.8, the usual power convention")
    for statistic, (color, linestyle) in styles.items():
        low, mean, high = (
            [read(experiment, label, d, statistic, f"detection_rate_{c}") for d in DURATIONS]
            for c in ("min", "mean", "max")
        )
        ax.fill_between(DURATIONS, low, high, color=color, alpha=0.18, zorder=2)
        ax.plot(DURATIONS, mean, color=color, ls=linestyle, lw=2.0, marker="o", ms=5, zorder=3,
                label=NAMES[statistic])
    ax.set_xscale("log")
    ax.set_xticks(DURATIONS)
    ax.set_xticklabels([f"{d:.0f}" for d in DURATIONS])
    ax.minorticks_off()
    ax.set_xlabel("record duration (s)")
    ax.set_ylim(-0.03, 1.03)
    ax.set_title(title, pad=10)
axes[0].set_ylabel("detection rate")
axes[1].legend(loc="center right", fontsize=8)

fig.suptitle("How long a record each statistic needs, on the bench", y=1.0)
fig.tight_layout()
plt.show()

**Read the figure.** Each line is the mean detection rate over the five repetitions, each shaded band runs from the smallest to the largest, and the dotted line marks 0.8, a convention for a reliable test and nothing more. On the left, the second-harmonic PLV crosses it between 200 s and 800 s, and the two rich quantum measures between 800 s and 3,200 s. On the right, at equal PLV, only the second-harmonic PLV crosses it. The rich measures begin to lift off the blind ones at 3,200 s, with ranges that still overlap; whether they would cross on longer records, the bench did not measure.

## 4. Equal moduli, different relative phase

Two complex moments hold more than two moduli. Picture the law as a flat level plus two waves around the circle: one with a single crest, as tall as $|m_1|$ makes it, and one with two crests, as tall as $|m_2|$ makes it. The moduli do not say where the two-crest wave sits along the one-crest wave, and that is one more number:

$$ \beta \;=\; \arg m_2 \;-\; 2 \arg m_1 . $$

It is the **relative phase** of the two harmonics. Shifting every phase difference by a constant slides both waves together and leaves $\beta$ unchanged. At $\beta = 0$ a crest of the second wave sits on the crest of the first, and the two together are their own mirror image. Otherwise the second wave is slid to one side and the law **leans**; mirror the law, $\Delta \to -\Delta$, and $\beta$ changes sign. In `15` both moments were real and positive, so $\beta$ was zero and stayed out of sight.

The classical statistic that reads the lean is the second sine moment about the mean direction $\mu = \arg m_1$:

$$ \bigl\langle \sin 2(\Delta - \mu) \bigr\rangle \;=\; |m_2| \sin\beta . $$

Pewsey (2002) built a test of reflective symmetry on a sine moment of this kind, taken about an estimate of the central direction. The bench solved for two settings, A and B, with the same $|m_1|$ **and** the same $|m_2|$, so that only $\beta$ differs. Let us look at them, and compute the sine by hand on fresh records.

In [ ]:
MODULI_LABEL = "k1_ref=0.05,k2_ref=0.12"
setting_a, setting_b = solved("equal_moduli", MODULI_LABEL)

fig, ax = plt.subplots(figsize=(7.5, 4.4))
print(f"{'':<3}{'K1':>8}{'K2':>9}{'|m1|':>9}{'|m2|':>9}{'beta':>10}{'cos(beta)':>11}"
      f"{'|m2| sin(beta)':>16}{'sine, your records':>21}")
for tag, (k1, k2), color in (("A", setting_a, COLORS["source"]),
                             ("B", setting_b, COLORS["target"])):
    m = exact_moments(k1, k2, max_harmonic=30)
    mu = np.angle(m[1])                               # mean direction of the law
    beta = np.angle(m[2] * np.conj(m[1]) ** 2)        # arg m2 - 2 arg m1, kept in (-pi, pi]

    # The second sine moment by hand, on fresh records: each record is turned back by its
    # own mean direction, then sin(2 x) is averaged over its samples.
    delta = simulate(k1, k2)
    mu_record = np.angle(np.exp(1j * delta).mean(axis=1))[:, None]
    sine = np.sin(2.0 * (delta - mu_record)).mean(axis=1)

    print(f"{tag:<3}{k1:>8.4f}{k2:>+9.4f}{abs(m[1]):>9.4f}{abs(m[2]):>9.4f}"
          f"{np.degrees(beta):>+8.1f}°{np.cos(beta):>11.3f}"
          f"{abs(m[2]) * np.sin(beta):>+16.3f}{mean_and_error(sine):>21}")

    # The exact law, turned back by its mean direction so that both laws are centred alike.
    ax.plot(angle, law_from_moments(m, angle + mu), color=color, lw=2.0,
            label=rf"{tag}: $\beta = {np.degrees(beta):+.1f}°$")
ax.axvline(0.0, color=COLORS["muted"], lw=1.0, ls=":")
ax.set_xlabel(r"phase difference, counted from the mean direction (rad)")
ax.set_ylabel("probability density (1/rad)")
ax.set_xticks([-np.pi, -np.pi / 2, 0.0, np.pi / 2, np.pi])
ax.set_xticklabels([r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"])
ax.set_title("Same two moduli, two different laws", pad=10)
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

**Read the output and the figure.** The sine you computed by hand matches $|m_2| \sin\beta$: $0.078 \pm 0.004$ for $+0.080$ in A, $-0.211 \pm 0.005$ for $-0.209$ in B.

In the figure both exact laws are turned so that their mean direction sits at zero. They share their two moduli to four decimals, and they are two different laws: A leans to the right of its mean direction, B to the left and further. That lean is what $\beta$ measures, $+9.9°$ against $-26.7°$. No law of this bench is its own mirror image, because the two oscillators have different natural frequencies: the first-harmonic setting of section 1 already has $\beta = +17°$.

In [ ]:
NAMES["relphase_sin"] = "sine"     # the bench's name for |m2| sin(beta)
NAMES["relphase_cos"] = "cosine"   # and for |m2| cos(beta)

print("Equal moduli, records of 3,200 s")
print(f"{'statistic':<12}{'exact A':>10}{'exact B':>10}{'area':>8}{'detection rate':>16}")
for statistic in ("plv", "plv2", "relphase_sin", "relphase_cos", "qmi_naive", "qmi_rich",
                  "bures_rich"):
    cells = [read("equal_moduli", MODULI_LABEL, 3200.0, statistic, column)
             for column in ("exact_discrete_a", "exact_discrete_b", "auc_mean",
                            "detection_rate_mean")]
    print(f"{NAMES[statistic]:<12}{cells[0]:>10.4f}{cells[1]:>10.4f}"
          f"{cells[2]:>8.2f}{cells[3]:>16.2f}")

print()
print("Detection rate of the sine at the three record durations: " + ", ".join(
    f"{read('equal_moduli', MODULI_LABEL, d, 'relphase_sin', 'detection_rate_mean'):.2f}"
    f" at {d:,.0f} s" for d in DURATIONS))

**Read the output.** This is the verdict of the bench on that pair. PLV and the second-harmonic PLV are at chance, as built. The sine separates the two settings perfectly at 3,200 s: an AUC of 0.00 means that B is lower than A in every pair of records, as its exact values say. It needs those long records: 0.23 at 200 s, 0.64 at 800 s. The cosine, $|m_2| \cos\beta$, separates them weakly: an AUC of 0.17 and a detection rate of 0.25.

The quantum measures do not separate them: an AUC of 0.51 and a detection rate of 0.04. Yet the exact rich QMI is not the same in A and B: 0.0799 against 0.0807 nat. So it does read something of $\beta$, a gap of 0.0008 nat that records of 3,200 s do not show. Let us find out what it reads, by turning $\beta$ by hand at fixed moduli.

In [ ]:
def exact_measures(modulus_1, modulus_2, beta):
    """Exact long-run statistics of a law whose first two moments have these moduli and this
    relative phase beta (rad).

    Only two moments are needed: the joint state of the qutrit embedding holds nothing else
    of the law, which is the premise of this whole section.
    """
    moments = np.array([1.0, modulus_1, modulus_2 * np.exp(1j * beta)], dtype=complex)
    return long_run_statistics(moments, weight_sets={})


m_a = exact_moments(*setting_a)
modulus_1, modulus_2 = abs(m_a[1]), abs(m_a[2])   # the moduli shared by A and B

print(f"|m1| = {modulus_1:.4f} and |m2| = {modulus_2:.4f} held fixed, "
      "relative phase turned by hand")
print(f"{'beta (deg)':>11}{'|m2| sin':>10}{'|m2| cos':>10}"
      f"{'naive QMI':>11}{'rich QMI':>10}{'rich Bures':>12}")
for degrees in (-180, -90, -30, 0, 30, 90, 180):
    s = exact_measures(modulus_1, modulus_2, np.radians(degrees))
    # Adding 0.0 after rounding turns a "-0.000" into "0.000" on display.
    sine, cosine = round(s["relphase_sin"], 3) + 0.0, round(s["relphase_cos"], 3) + 0.0
    print(f"{degrees:>11d}{sine:>10.3f}{cosine:>10.3f}"
          f"{s['qmi_naive']:>11.5f}{s['qmi_rich']:>10.5f}{s['bures_rich']:>12.5f}")

# Why: turning beta into -beta replaces each moment by its complex conjugate, and the joint
# state by its complex conjugate, which has the same eigenvalues.
moments = np.array([1.0, modulus_1, modulus_2 * np.exp(0.5j)], dtype=complex)
rho, rho_mirror = long_run_joint_density(moments), long_run_joint_density(np.conj(moments))
print()
print("joint state of the mirrored law = complex conjugate of the joint state:",
      np.allclose(rho_mirror, np.conj(rho)))
print("largest difference between their eigenvalues:",
      f"{np.abs(np.linalg.eigvalsh(rho_mirror) - np.linalg.eigvalsh(rho)).max():.1e}")

def is_a_law(m1, m2):
    """Whether (1, m1, m2) can be the first moments of a probability law on the circle.

    The condition is that their Toeplitz matrix has no negative eigenvalue.
    """
    toeplitz = np.array([[1.0, np.conj(m1), np.conj(m2)],
                         [m1, 1.0, np.conj(m1)],
                         [m2, m1, 1.0]])
    return np.linalg.eigvalsh(toeplitz).min() >= 0.0


# The same check on pairs of moments drawn at random, keeping those a law can have.
rng = np.random.default_rng(16)
largest, n_laws = 0.0, 0
for _ in range(2000):
    r1, r2, beta = rng.uniform(0.02, 0.6), rng.uniform(0.02, 0.6), rng.uniform(-np.pi, np.pi)
    if not is_a_law(r1, r2 * np.exp(1j * beta)):
        continue
    n_laws += 1
    plus, minus = exact_measures(r1, r2, beta), exact_measures(r1, r2, -beta)
    largest = max(largest, *(abs(plus[key] - minus[key])
                             for key in ("qmi_naive", "qmi_rich", "bures_rich")))
print("largest change of a quantum measure when beta becomes -beta, "
      f"over {n_laws:,} random pairs of moments: {largest:.1e}")

**Read the output.** Read the table from its middle row outwards. The naive QMI does not move: it reads $|m_1|$ and nothing else. The rich QMI and the rich Bures distance do move with $\beta$, from 0.07975 nat at $0°$ to 0.09839 at $180°$. And they take **exactly the same value at $+\beta$ and $-\beta$**, row for row.

The lines below the table say why. Changing the sign of $\beta$ mirrors the law, which replaces each moment, and with it the joint state, by its complex conjugate. The QMI is made of entropies, which depend on eigenvalues, and a matrix and its conjugate have the same eigenvalues. The Bures distance compares the joint state with the product of its marginals, and both are conjugated together. Over 1,961 random pairs of moments the largest change is zero to the last digit.

So these measures read the relative phase **through its cosine only**. The sign of $\beta$ — which way the law leans — is invisible to them, and the sine reads exactly that. On the pair of the bench the sign contributes nothing to their gap. What is left is the difference in size, $9.9°$ against $26.7°$: 0.0008 nat, where the classical cosine itself separates the pair only weakly.

In [ ]:
fig, (ax_modulus, ax_phase) = plt.subplots(1, 2, figsize=(12, 4.8))

# (a) Rich QMI against the second modulus, at the first moment of notebook 15 (0.4).
FIRST_MOMENT_15 = 0.4
second = np.linspace(0.0, 0.45, 181)


def rich_qmi(modulus_2, beta, modulus_1=FIRST_MOMENT_15):
    return exact_measures(modulus_1, modulus_2, beta)["qmi_rich"]


lines = [(0.0, COLORS["quantum"], "-", r"$\beta = 0$"),
         (np.pi / 2, COLORS["muted"], "--", r"$\beta = \pm 90°$"),
         (np.pi, COLORS["target"], "-", r"$\beta = 180°$")]
for beta, color, linestyle, label in lines:
    ax_modulus.plot(second, [rich_qmi(a2, beta) for a2 in second],
                    color=color, ls=linestyle, lw=2.0, label=label)
start = rich_qmi(0.0, 0.0)
ax_modulus.axhline(start, color=COLORS["text"], lw=0.9, ls=":", zorder=1,
                   label=r"value at $|m_2| = 0$")
ax_modulus.scatter([0.0, 0.3], [start, rich_qmi(0.3, 0.0)], s=90, color=COLORS["highlight"],
                   edgecolor="white", linewidth=1.2, zorder=5,
                   label="the two ensembles of notebook 15")
ax_modulus.set_xlabel(r"$|m_2|$, with $|m_1| = 0.4$")
ax_modulus.set_ylabel("exact rich QMI (nats)")
ax_modulus.set_ylim(0.058, 0.116)   # room under the curves for the legend
ax_modulus.set_title("Against the second modulus", pad=10)
ax_modulus.legend(loc="lower right", fontsize=8, ncol=2)

# A closer look at the start of the in-phase curve, where the change is small.
in_phase = np.array([rich_qmi(a2, 0.0) for a2 in second])
near = second <= 0.2
zoom = ax_modulus.inset_axes([0.09, 0.58, 0.40, 0.34])
zoom.plot(second[near], in_phase[near] - start, color=COLORS["quantum"], lw=2.0)
zoom.axhline(0.0, color=COLORS["text"], lw=0.9, ls=":")
zoom.set_title(r"$\beta = 0$: change from $|m_2| = 0$ (nats)", fontsize=8)
zoom.tick_params(labelsize=7)

lowest = second[np.argmin(in_phase)]
back = second[(second > lowest) & (in_phase >= start)][0]
print(f"beta = 0, |m1| = 0.4: the rich QMI is lowest at |m2| = {lowest:.3f} "
      f"({start - in_phase.min():.4f} nat below its value at |m2| = 0) "
      f"and is back to that value at |m2| = {back:.3f}")
for beta, name in ((0.0, "beta = 0"), (np.pi, "beta = 180 deg")):
    print(f"{name}: at |m2| = 0.3 the rich QMI is {rich_qmi(0.3, beta) - start:+.4f} nat "
          "away from its value at |m2| = 0")

# (b) Rich QMI against the relative phase, at the moduli of the pair of section 4.
betas = np.linspace(-np.pi, np.pi, 241)
ax_phase.plot(np.degrees(betas), [rich_qmi(modulus_2, b, modulus_1) for b in betas],
              color=COLORS["quantum"], lw=2.0)
for tag, setting in (("A", setting_a), ("B", setting_b)):
    m = exact_moments(*setting)
    beta = np.angle(m[2] * np.conj(m[1]) ** 2)
    value = rich_qmi(abs(m[2]), beta, abs(m[1]))
    ax_phase.scatter(np.degrees(beta), value, s=90, color=COLORS["highlight"],
                     edgecolor="white", linewidth=1.2, zorder=5)
    ax_phase.annotate(tag, (np.degrees(beta), value), xytext=(0, 10),
                      textcoords="offset points", ha="center", color=COLORS["text"])
ax_phase.set_xlabel(r"relative phase $\beta$ (degrees)")
ax_phase.set_ylabel("exact rich QMI (nats)")
ax_phase.set_xticks([-180, -90, 0, 90, 180])
ax_phase.set_title("Against the relative phase", pad=10)

fig.suptitle("What the rich quantum mutual information does with the second moment", y=1.0)
fig.tight_layout()
plt.show()

**Read the figure.** The left panel answers the Challenge of `15`. It shows the exact rich QMI against the second modulus, at the first moment of `15` (0.4). The two rose dots are the two ensembles of `15`, and the blue curve through them is the family you explored there. Look at the inset: the curve first goes **down**. It is lowest at $|m_2| = 0.085$ and is back at its starting value, the dotted line, near $0.17$. With a high ensemble at $a_2 = 0.17$, `15` would have found a rich QMI as blind as PLV, beside a second-harmonic PLV reading a gap of 0.17. At $a_2 = 0.3$ the gap is 0.0049 nat, which is why so many samples were needed.

The amber curve is the same family with $\beta = 180°$. It rises from the start, and at $|m_2| = 0.3$ its gap is 0.0187 nat, nearly four times larger.

On the right, the exact rich QMI against $\beta$ at the moduli of the bench pair. The curve is its own mirror image, lowest at $0°$ and highest at $\pm 180°$, a swing of 0.02 nat on 0.08. A and B sit near its flat bottom, almost level. The rich QMI is a function of $|m_1|$, $|m_2|$ and $\cos\beta$ together, and how well it competes with a second-harmonic PLV depends on the relative phase.

## 5. What a quantum measure reads, and what it does not

You now hold everything the accounting needs. Take it in four sentences, each with its evidence.

**It reads past PLV, when the embedding carries the second harmonic.** With a second harmonic alone the exact PLV is zero, and the rich QMI detects the coupling with a rate of 0.97 to 1.00 at 3,200 s. At equal PLV its exact value differs. The naive embedding's QMI follows PLV in every table.

**It reads as far as a classical statistic of the same two harmonics, and no further.** This is a statement about content: in the long run every measure of this notebook is a function of $m_1$ and $m_2$, which classical statistics read directly. On finite records the equal-weight embedding was the slower reader: the second-harmonic PLV became reliable on shorter records in every detection comparison, and at equal PLV the rich QMI did not become reliable within 3,200 s.

**On the relative phase it reads less.** It sees $\cos\beta$ and is blind to the sign of $\beta$, which a classical sine moment detects.

**What quantum optimal transport brings is one common geometry.** The classical side is a menu of statistics, each chosen by hand. The quantum side builds one state and measures its distance from independence, by a relative entropy (QMI) or a transport distance (Bures); choosing a statistic becomes choosing an embedding. That is a unification. On this bench it is not a gain in sensitivity.

The capstone adds four rows to the classical-to-quantum dictionary, gathered here from `13` to `16`:

| Classical | Quantum | What the capstone measured |
|---|---|---|
| PLV, $\lvert m_1\rvert$ | the first-harmonic coherence of $\rho_{AB}$ | the naive QMI is a function of PLV alone (`13`) |
| second-harmonic PLV, $\lvert m_2\rvert$ | the second-harmonic coherence of the qutrit state | with equal weights, the rich QMI and Bures distance respond to it and need longer records (`15`, `16`) |
| second sine moment, $\lvert m_2\rvert \sin\beta$ | none among the measures computed here | QMI and Bures distance are even in $\beta$: they read $\cos\beta$ only (`16`) |
| choosing a statistic | choosing an embedding | the embedding sets what the measure can read (`13`, `14`) |

The limits are part of the result. The bench is one model — the same coupling in both directions, independent noises of equal size, one noise level, one frequency difference — with records up to 3,200 s and a handful of settings; the 0.8 line is a convention. The reading of the relative phase rests on one pair and on the exact computation, and $\beta$ has no value where $|m_1|$ is close to zero. One stronger setting ($K_2 = 0.3$), whose records do not explore the long-run law within these durations, is kept apart in the files. The finite-record race set equal weights against one classical statistic: the same files hold an embedding with unequal weights that keeps pace with the second-harmonic PLV, and `17` is about it. And the blindness to the sign of $\beta$ belongs to measures of this kind: the expectation value of a well-chosen observable could in principle read it.

## Your turn

**Warm-up.** At $K_2 = 0.1$ on records of 200 s, the rich QMI has a detection rate of 0.32. Guess its AUC, then read it with `read("detection", "k2=0.1", 200.0, "qmi_rich", "auc_mean")`. What does a high AUC beside a low detection rate tell you about the size of the shift, compared with the scatter of one record?

**Go further.** Section 5 says that an embedding with unequal weights keeps pace with the second-harmonic PLV. Check it: the files call it `qmi_weighted_02`. Give it a name, `NAMES["qmi_weighted_02"] = "weighted"`, then call `print_rates("equal_plv", EQUAL_PLV_LABELS, ["plv2", "qmi_rich", "qmi_weighted_02"])`. Where does it stand beside the two others? Keep your answer for `17`.

**Challenge.** With `exact_measures`, compare two laws that differ only in the sign of the relative phase ($+20°$ against $-20°$), then two that differ in its size ($+20°$ against $+60°$), at the moduli of section 4. You should find a rich QMI gap of exactly zero in the first pair and a gap that is not zero in the second. Which pair could a rich QMI separate with long enough records, and which can it never separate? Which classical statistic of section 4 would you use for each, the sine (`relphase_sin`) or the cosine (`relphase_cos`)?

## What you built

You closed the capstone on measurements, and on a system nobody tuned by hand. That is a careful piece of work, and it is yours.

- You simulated a **two-harmonic dyad**, matched its sample moments to exact values, and **reproduced a comparison** of a committed bench with a new seed.
- You found that the richer quantum measures **detect a coupling PLV cannot see**, and that at equal PLV they differ in the long run without becoming reliable within 3,200 s.
- You showed, exactly, that they read the relative phase of the two harmonics **through its cosine only**.
- You read every result on three levels — the exact long-run value, the area under the curve, the detection rate — and said which one each sentence rests on. That habit will serve you with any measure you meet.

State it precisely, and no further. **A quantum coupling measure reads past the phase-locking value when its embedding carries more than the first harmonic. It reads as far as a classical statistic of the same harmonics and no further, and less far on their relative phase. With equal weights it needed longer records than the second-harmonic PLV. What it offers is one geometry in which those statistics become one distance and one choice of embedding.**

## What's next

The embedding is the dial, so the next notebook turns it: `17_embedding_design_amplifies_the_effect` changes the weights of the harmonics and puts the question to the same protocol. `18_what_no_classical_data_embedding_can_buy` finds the structural boundary of the programme. `19_vqe_limitation_de_palma` turns the geometry towards a proven result on quantum algorithms, and `20_frontier_synthesis` gathers the module.

## References

- H. Haken, J. A. S. Kelso & H. Bunz, "A theoretical model of phase transitions in human hand movements", *Biological Cybernetics* **51**, 347–356 (1985). DOI:10.1007/BF00336922 — the two-harmonic coupling of the relative phase.
- G. Schöner, H. Haken & J. A. S. Kelso, "A stochastic theory of phase transitions in human hand movement", *Biological Cybernetics* **53**, 247–257 (1986). DOI:10.1007/BF00336995 — the same model with noise.
- J.-P. Lachaux, E. Rodriguez, J. Martinerie & F. J. Varela, "Measuring phase synchrony in brain signals", *Human Brain Mapping* **8**, 194–208 (1999). DOI:10.1002/(SICI)1097-0193(1999)8:4<194::AID-HBM4>3.0.CO;2-C — the phase-locking value.
- K. V. Mardia & P. E. Jupp, *Directional Statistics* (Wiley, 2000). DOI:10.1002/9780470316979 — circular moments of a phase distribution.
- A. Pewsey, "Testing circular symmetry", *Canadian Journal of Statistics* **30**(4), 591–600 (2002). DOI:10.2307/3316098 — a test of reflective symmetry built on the second sine moment about an estimate of the central direction.
- J. A. Hanley & B. J. McNeil, "The meaning and use of the area under a receiver operating characteristic (ROC) curve", *Radiology* **143**, 29–36 (1982). DOI:10.1148/radiology.143.1.7063747 — the area under the curve as a probability of correct ranking.
- M. M. Wilde, *Quantum Information Theory*, 2nd ed. (Cambridge University Press, 2017). DOI:10.1017/9781316809976 — quantum mutual information.
- D. Trevisan, "Quantum optimal transport: an invitation", *Bollettino dell'Unione Matematica Italiana* **18**, 347–360 (2025). DOI:10.1007/s40574-024-00428-5 — the survey of quantum optimal-transport formulations behind this module.

**Previous:** `notebooks/04_QuantumOptimalTransport/15_discriminating_experiment.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/17_embedding_design_amplifies_the_effect.ipynb`